In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- bench_interp_cols ---
FIX_BENCH_INTERP_COLS_REQUEST_PD = SimpleNamespace(df=pd.DataFrame({"station_id":["01048"],"latitude":[48.0],"longitude":[11.5],"date":["2020-01-01"],"value":[5.2]}))
FIX_BENCH_INTERP_COLS_REQUEST_PL = SimpleNamespace(df=pl.from_pandas(FIX_BENCH_INTERP_COLS_REQUEST_PD.df))

# --- bench_interp_filter ---
FIX_BENCH_INTERP_FILTER_DAY_TIME = pd.Timestamp("2020-01-01")
DF_BENCH_INTERP_FILTER_PD = pd.DataFrame({"station_id": ["A","B","C"], "latitude": [48.1,48.2,48.3], "longitude": [11.5,11.6,11.7], "date": ["2020-01-01 00:00:00+00:00","2020-01-02 00:00:00+00:00","2020-01-03 00:00:00+00:00"], "value": [1.0,2.0,3.0]})
DF_BENCH_INTERP_FILTER_PL = pl.from_pandas(DF_BENCH_INTERP_FILTER_PD)
df = DF_BENCH_INTERP_FILTER_PD

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_bench_interp_cols(request):
    station_ids = request.df["station_id"].values.tolist()
    latitudes = request.df["latitude"].values.tolist()
    longitudes = request.df["longitude"].values.tolist()
    return longitudes

def before_bench_interp_filter(day_time):
    filtered_df = df[df["date"].astype(str).str[:] == day_time.strftime("%Y-%m-%d %H:%M:%S+00:00")]
    values = filtered_df["value"].values.tolist()
    return values

In [ ]:
# ── Generated wrappers (experiment-generated Polars) ─────────────────────────

def gen_bench_interp_cols(request):
    station_ids = request.df["station_id"].to_list()
    latitudes = request.df["latitude"].to_list()
    longitudes = request.df["longitude"].to_list()
    return longitudes

def gen_bench_interp_filter(day_time):

    filtered_df = df.filter(
        pl.col("date").cast(pl.Utf8) == day_time.strftime("%Y-%m-%d %H:%M:%S+00:00")
    )
    values = filtered_df["value"].to_list()
    return values

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def _comparison_label(label):
    text = str(label)
    if text.lstrip().startswith(("L2", "L3")):
        return text
    return f"L2 equivalence {text}"

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: bench_interp_filter ===

# L1 smoke – generated
try:
    df = DF_BENCH_INTERP_FILTER_PL
    _r = gen_bench_interp_filter(FIX_BENCH_INTERP_FILTER_DAY_TIME)
    print("✅ L1 smoke gen_bench_interp_filter: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_bench_interp_filter: {type(_e).__name__}: {_e}")
finally:
    df = DF_BENCH_INTERP_FILTER_PD

# L1 smoke – before
try:
    df = DF_BENCH_INTERP_FILTER_PD
    _rb = before_bench_interp_filter(FIX_BENCH_INTERP_FILTER_DAY_TIME)
    print("✅ L1 smoke before_bench_interp_filter: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_bench_interp_filter: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    df = DF_BENCH_INTERP_FILTER_PD
    _rb = before_bench_interp_filter(FIX_BENCH_INTERP_FILTER_DAY_TIME)
    df = DF_BENCH_INTERP_FILTER_PL
    _rg = gen_bench_interp_filter(FIX_BENCH_INTERP_FILTER_DAY_TIME)
    if _rb == _rg:
        print("✅ L2 equivalence bench_interp_filter: MATCH")
    else:
        print(f"❌ L2 equivalence bench_interp_filter: MISMATCH — before={_rb!r}, gen={_rg!r}")
except Exception as _e:
    print(f"❌ L2 equivalence bench_interp_filter: setup error — {type(_e).__name__}: {_e}")
finally:
    df = DF_BENCH_INTERP_FILTER_PD

# AUDIT-50: compare no-match results on both sides.
try:
    df = DF_BENCH_INTERP_FILTER_PD; _rb = before_bench_interp_filter(pd.Timestamp("1999-01-01"))
    df = DF_BENCH_INTERP_FILTER_PL; _rg = gen_bench_interp_filter(pd.Timestamp("1999-01-01"))
    if _rb == _rg == []: print("✅ L3 edge bench_interp_filter no match oracle: MATCH")
    else: print(f"❌ L3 edge bench_interp_filter no match oracle: MISMATCH — before={_rb}, gen={_rg}")
except Exception as _e:
    print(f"❌ L3 edge bench_interp_filter no match oracle: {type(_e).__name__}: {_e}")
finally:
    df = DF_BENCH_INTERP_FILTER_PD
